# Phase 8: Long-term memory and the Drafter

Two things are new. **Long-term memory** keeps what reviewers corrected, so similar reports get hints next time. The **Drafter** writes a suggested reply, and a person approves it before it is ready to send. Nothing is ever sent without getting human approval.

Memory here is hints in the prompt, not model training, so it nudges the model but does not guarantee a change.

In [1]:
import json
from uuid import uuid4

from langgraph.types import Command

from community_voice.agents.triage import triage
from community_voice.db.repository import get_draft, reset_database
from community_voice.graph.memory import (
    NAMESPACE,
    find_hints,
    get_store,
    remember_correction,
    reset_memory,
)
from community_voice.graph.persistence import get_checkpointer
from community_voice.graph.workflow import build_graph
from community_voice.rag.reports import reset_reports
from community_voice.schemas.report import ReportIn
from community_voice.schemas.review import ReviewDecision
from community_voice.schemas.triage import TriageResult

## 1. Start fresh

The next cell creates the memory file, `data/memory.db`, and clears old memory. It uses the local embedding model, so the first run may download it.

In [2]:
store = get_store()
reset_memory(store)
reset_database()
reset_reports()

## 2. Triage without memory

A school with no water. A model would normally call this `high`, not `critical`.

In [3]:
new_report = ReportIn(
    text="The taps at Sizwe Primary School in Gqeberha have been dry since Monday."
)
before = triage(new_report)
print(before.urgency.value, "-", before.urgency_reason)

high - A school lacking water for several days is an essential service issue affecting many children.


## 3. A reviewer corrects a similar report

We act out an earlier case. Triage said `high`, and the reviewer made it `critical`, because in this municipality a school without water is always critical. Only real changes are remembered.

In [4]:
earlier_text = (
    "There has been no water at Thandi Primary School in Soshanguve for three days."
)
ai_said = TriageResult(
    language="en",
    english_text=earlier_text,
    category="water",
    summary="No water at a primary school.",
    urgency="high",
    urgency_reason="Essential service out for days.",
    area_mentioned="Soshanguve",
)
reviewer = ReviewDecision(
    action="approve",
    urgency="critical",
    note="A school without water is always critical here.",
)
saved = remember_correction(
    store, "r10", earlier_text, ai_said, "Tshwane Water and Sanitation", reviewer
)
print("Remembered:", saved)

Remembered: True


## 4. Hints for a similar report

Search by meaning. A higher score is a closer match. An unrelated report is included to show what a weak match scores.

In [5]:
unrelated = "A large pothole on Main Road in Woodstock damaged my tyre."
for query in (new_report.text, unrelated):
    print(query[:50])
    for item in store.search(NAMESPACE, query=query, limit=1):
        print("  ", round(item.score, 2), item.value["summary"])
hints = find_hints(store, new_report.text)
print(hints)

The taps at Sizwe Primary School in Gqeberha have 
   0.36 No water at a primary school.
A large pothole on Main Road in Woodstock damaged 
   0.06 No water at a primary school.
['Similar report (No water at a primary school.): a reviewer changed urgency from high to critical. Note: A school without water is always critical here.']


## 5. Triage again, with the hints

In [6]:
after = triage(new_report, hints=hints)
print(after.urgency.value, "-", after.urgency_reason)

critical - Lack of water at a primary school is a health and hygiene risk for learners and staff.


## 6. The Drafter and the second approval

The critical Tembisa report skips the web check, so this path is predictable. The graph pauses twice: first to review the report, then to review the drafted reply.

In [4]:
graph = build_graph(get_checkpointer(), store)
report_id = f"r05-{uuid4().hex[:6]}"
config = {"configurable": {"thread_id": report_id}}
inputs = {
    "report_id": report_id,
    "report": ReportIn(
        text="After last night's storm there are live wires hanging low over "
        "the road near Tembisa Hospital. Children walk under them to get to "
        "school."
    ),
}
result = graph.invoke(inputs, config, version="v2")
print("Paused at:", graph.get_state(config).next)

Paused at: ('human_review',)


The reviewer approves the report. The Drafter then writes a reply and the graph pauses again.

Read the draft against the policy files before approving it. The second approval exists to catch mistakes.

In [5]:
result = graph.invoke(Command(resume={"action": "approve"}), config, version="v2")
print("Paused at:", graph.get_state(config).next)
print(json.dumps(result.interrupts[0].value["draft"], indent=2))

Paused at: ('draft_review',)
{
  "body": "Dear resident,\n\nThank you for reporting the low-hanging live electrical wires near Tembisa Hospital (Reference: r05-75c64b). We treat this issue as a critical danger to public safety.\n\nIn accordance with our Electricity Service Charter, a crew will arrive at the site within 2 hours. Please keep away from the area and warn others to do the same.\n\nWe have acknowledged your report within the required 1-hour timeframe for critical issues. You may also contact the Ekurhuleni Electricity department at 000 000 0005 for further updates.\n\nSincerely,\n\nMunicipality of Ekurhuleni",
  "policy_sources": [
    "electricity_charter.md",
    "sla_rules.md"
  ]
}


## 7. Approve the draft with an edit

In [6]:
draft = result.interrupts[0].value["draft"]["body"]
edited = draft + "\n\nPlease keep children away from the wires."
answer = {"action": "approve", "text": edited}
result = graph.invoke(Command(resume=answer), config, version="v2")
print("Status:", result.value["status"])
print(get_draft(report_id)["body"])

Status: ready_to_send
Dear resident,

Thank you for reporting the low-hanging live electrical wires near Tembisa Hospital (Reference: r05-75c64b). We treat this issue as a critical danger to public safety.

In accordance with our Electricity Service Charter, a crew will arrive at the site within 2 hours. Please keep away from the area and warn others to do the same.

We have acknowledged your report within the required 1-hour timeframe for critical issues. You may also contact the Ekurhuleni Electricity department at 000 000 0005 for further updates.

Sincerely,

Municipality of Ekurhuleni

Please keep children away from the wires.


## What to notice

- Memory is shared across reports. Short-term memory belongs to one report.
- Only a real change by a reviewer is remembered.
- The Drafter works from our policy documents, but it can still pick the wrong figure. A person checks the draft.
- A person approves twice before a reply is ready to send.